In [1]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import tabulate
import json
import csv
import os
import sys

In [ ]:
#Return file path with year number
def base_file(n:int) -> str:
        try:
            return f'../DATA/Bronce/COVID19MEXICO202{n}/COVID19MEXICO202{n}.csv'
        except FileNotFoundError:
              return f'[Error] date not found'

In [ ]:
#Count number of rows in file
def count_rows(file_path: str, chunk_size: int = 1024 * 1024) -> int:
    rows = 0
    try:
        with open(file_path, 'rb') as f:
            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break
                rows += chunk.count(b'\n')
        return rows
    except FileNotFoundError:
        print(f"[Error] File not found: {file_path}")
        return 0

In [ ]:
#Count number of rows for file
for i in range(0,4):
    b = base_file(i)
    print(f'{b} to: {count_rows(b):,}')

COVID19MEXICO2020/COVID19MEXICO2020.csv to: 3,868,397
COVID19MEXICO2021/COVID19MEXICO2021.csv to: 8,830,346
COVID19MEXICO2022/COVID19MEXICO2022.csv to: 6,451,945
COVID19MEXICO2023/COVID19MEXICO2023.csv to: 1,222,220


In [2]:
# View general struct of part of the one file
def view_struct_chunk_csv(file_path: str, file_count: int = 100):
    print(f"Extraction {file_count} rows...\n")
    
    try:
        df = pd.read_csv(
            file_path,
            nrows=file_count,
            low_memory=False
        )
        
        # 1. Validation
        true_rows = len(df)
        print(f"OK. Load {true_rows} in RAM.\n")
        
        # 2. Metadata
        print("--- REPORT TEC ---")
        
        report = pd.DataFrame({
            'Type': df.dtypes,
            'Null values': df.isnull().sum(),
            'Porc_Nulls_(%)': (df.isnull().sum() / true_rows) * 100,
            'Bytes': df.memory_usage(deep=True).drop('Index')
        })
        
        print(report.to_string())
        
        # 3. Columns
        print("\n Columns avaible")
        print(f"{df.columns} \n")

        # 4. RAM
        total_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
        print(f"\n RAM usage: {total_mb:.4f} MB")
        
        return df
        
    except FileNotFoundError:
        print(f"{file_path} not found")
        sys.exit(1)
    except Exception as e:
        print(f" ERROR {e}")
        sys.exit(1) 

In [4]:
# View general struct of part of the 2020 file
view_struct_chunk_csv("../DATA/Bronce/COVID19MEXICO2024/COVID19MEXICO2024.csv",200)

Extraction 200 rows...

OK. Load 200 in RAM.

--- REPORT TEC ---
                            Type  Null values  Porc_Nulls_(%)  Bytes
FECHA_ACTUALIZACION          str            0             0.0  11800
ID_REGISTRO                  str            0             0.0  11197
ORIGEN                     int64            0             0.0   1600
SECTOR                     int64            0             0.0   1600
ENTIDAD_UM                 int64            0             0.0   1600
SEXO                       int64            0             0.0   1600
ENTIDAD_NAC                int64            0             0.0   1600
ENTIDAD_RES                int64            0             0.0   1600
MUNICIPIO_RES              int64            0             0.0   1600
TIPO_PACIENTE              int64            0             0.0   1600
FECHA_INGRESO                str            0             0.0  11800
FECHA_SINTOMAS               str            0             0.0  11800
FECHA_DEF                    str      

,FECHA_ACTUALIZACION,ID_REGISTRO,ORIGEN,SECTOR,ENTIDAD_UM,SEXO,ENTIDAD_NAC,ENTIDAD_RES,MUNICIPIO_RES,TIPO_PACIENTE,...,RESULTADO_PCR,RESULTADO_PCR_COINFECCION,TOMA_MUESTRA_ANTIGENO,RESULTADO_ANTIGENO,CLASIFICACION_FINAL_COVID,CLASIFICACION_FINAL_FLU,MIGRANTE,PAIS_NACIONALIDAD,PAIS_ORIGEN,UCI
0,2025-07-25,g9e51e0,1,15,21,2,21,21,114,2,...,5,5,2,97,7,7,99,México,97,2
1,2025-07-25,gaeb43c,1,12,9,1,19,9,12,1,...,34,998,2,97,3,7,99,México,97,97
2,2025-07-25,g944a2d,1,12,9,2,11,9,7,1,...,34,998,2,97,3,7,99,México,97,97
3,2025-07-25,g99409b,1,12,9,1,15,15,13,1,...,34,998,2,97,3,7,99,México,97,97
4,2025-07-25,ge8277c,1,4,15,2,15,15,81,1,...,5,5,2,97,7,7,99,México,97,97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2025-07-25,g6f415f,1,6,15,2,9,15,81,1,...,997,997,2,97,6,6,99,México,97,97
196,2025-07-25,gab8d1e,1,15,32,2,32,32,10,1,...,5,5,2,97,7,7,99,México,97,97
197,2025-07-25,ga98f68,1,15,32,1,32,32,20,2,...,5,5,2,97,7,7,99,México,97,2
198,2025-07-25,g5a87af,1,6,30,1,30,30,193,1,...,997,997,2,97,6,6,99,México,97,97


In [ ]:
#Create report of quality for file of different years
#Create list of error rows in files of different years
def quality_accurance(years: int, chunk_size: int = 100000):
    current_date = pd.to_datetime('today').normalize()
    min_date = pd.to_datetime('2018-01-01')
    
    date_columns = ['FECHA_INGRESO', 'FECHA_SINTOMAS', 'FECHA_DEF']
    columns_age = 'EDAD'
    
    dir_report = "./reportes_calidad/"
    dir_errors = "./datos_con_error/"

    for i in range(years):
        file_path = base_file(i) 
        base_name = os.path.basename(file_path)
        path_errors = os.path.join(dir_errors, f"errores_{base_name}")
        
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Proc: {file_path}")
        
        file_report = {
            "archivo_origen": file_path,
            "total_filas_evaluadas": 0,
            "total_filas_cuarentena": 0,
            "nulos_otras_columnas": {}
        }
        
        first_batch_error = True

        try:
            iterator_batches = pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)
            
            for batch in iterator_batches:
                file_report["total_filas_evaluadas"] += len(batch)
                
                global_mask_error = pd.Series(False, index=batch.index)
                
                batch['motivo_rechazo'] = ""

                for col in date_columns:
                    if col in batch.columns:
                        temp_dates = pd.to_datetime(batch[col], errors='coerce')
                        invalid_error = temp_dates.notna() & ((temp_dates < min_date) | (temp_dates > current_date))
                        
                        global_mask_error = global_mask_error | invalid_error
                        batch.loc[invalid_error, 'motivo_rechazo'] += f"[{col} fuera de rango]"

                if columns_age in batch.columns:
                    tmp_age = pd.to_numeric(batch[columns_age], errors='coerce')
                    age_mask = tmp_age.notna() & ((tmp_age < 0) | (tmp_age > 100))
                    
                    global_mask_error = global_mask_error | age_mask
                    batch.loc[age_mask, 'motivo_rechazo'] += f"[EDAD inválida]"

                corrupt_registr = batch[global_mask_error]
                
                if not corrupt_registr.empty:
                    file_report["total_filas_cuarentena"] += len(corrupt_registr)
                    
                    corrupt_registr.to_csv(
                        path_errors, 
                        mode='a', 
                        index=False, 
                        header=first_batch_error
                    )
                    first_batch_error = False 

                valid_columns = date_columns + [columns_age, 'motivo_rechazo']
                other_columns = [c for c in batch.columns if c not in valid_columns]
                
                for col in other_columns:
                    nulos_in_batch = int(batch[col].isnull().sum())
                    if nulos_in_batch > 0:
                        file_report["nulos_otras_columnas"][col] = file_report["nulos_otras_columnas"].get(col, 0) + nulos_in_batch

            json_path_report = os.path.join(dir_report, f"reporte_{base_name}.json")
            with open(json_path_report, 'w', encoding='utf-8') as f:
                json.dump(file_report, f, indent=4)
                

        except Exception as e:
            print(f"[Error]{file_path}: {e}")

### The cell below has already been completed
## ↓ ↓ ↓ ↓

In [ ]:
#Run qualyty report for five years (2020-2025) with 100000 elemts for chunk
#It has already been done
#quality_accurance(years=5, chunk_size=100000)

In [ ]:
#Listo of files in one folder for extension
def list_reports(path: str, extension: str) -> list:
    folder = Path(path)
    
    if not folder.exists() or not folder.is_dir():
        print(f"[Error] file not found: {path}")
        return []

    correct_file = []
    
    for path_file in folder.glob(f"*.{extension}"):
        if path_file.is_file():
            correct_file.append(str(path_file.resolve()))
    
    return correct_file

In [ ]:
# View QA reports
directory = "./reportes_calidad/"
list_of_jsons = list_reports(directory,"json")

for json_file in list_of_jsons:
    try:
        with open(json_file, 'r', encoding='utf-8') as file:
            raw_data = json.load(file)
        
        df_report = pd.json_normalize(raw_data)
    
        print(df_report.to_markdown(index=False))
        print("#############################")
        total_evaluate = float(raw_data.get('total_filas_evaluadas'))
        total_error = float(raw_data.get('total_filas_cuarentena'))
        print(f"Percentage of failure: {total_error/total_evaluate:.2%}")


    except FileNotFoundError:
        print(f"[Error] file not found : {json_file}")
    except json.JSONDecodeError as e:
        print(f"[Error] corrupt file: {e}")

| archivo_origen                          |   total_filas_evaluadas |   total_filas_cuarentena |
|:----------------------------------------|------------------------:|-------------------------:|
| COVID19MEXICO2024/COVID19MEXICO2024.csv |                  177618 |                       88 |
#############################
Percentage of failure: 0.05%
| archivo_origen                          |   total_filas_evaluadas |   total_filas_cuarentena |
|:----------------------------------------|------------------------:|-------------------------:|
| COVID19MEXICO2021/COVID19MEXICO2021.csv |                 8830345 |                     5271 |
#############################
Percentage of failure: 0.06%
| archivo_origen                          |   total_filas_evaluadas |   total_filas_cuarentena |
|:----------------------------------------|------------------------:|-------------------------:|
| COVID19MEXICO2020/COVID19MEXICO2020.csv |                 3868396 |                     1374 |
#########

In [ ]:
# Remove error rows in data files
def clean_csv_for_batch(input_path: str, output_path: str, remove_list: list, chunk_size: int = 100000):
    
    is_first_batch = True 
    row_procesed = 0
    row_conserved = 0
    
    try:
        iterator_batch = pd.read_csv(input_path, chunksize=chunk_size, low_memory=False,quoting=csv.QUOTE_NONE)
        
        for chunk in iterator_batch:
            row_procesed += len(chunk)
            
            clean_chunk = chunk[~chunk["ID_REGISTRO"].isin(remove_list)]
            
            if clean_chunk.empty:
                continue
                
            write_mode = 'w' if is_first_batch else 'a'
            
            clean_chunk.to_csv(
                output_path, 
                mode=write_mode, 
                index=False, 
                header=is_first_batch
            )
            
            row_conserved += len(clean_chunk)
            is_first_batch = False
        print(f"Clean file: {input_path}")
        
    except FileNotFoundError:
        print(f"[Error] file not found: {input_path}")
    except Exception as e:
        print(f"[Error] {e}")

### The cell below has already been completed
## ↓ ↓ ↓ ↓

In [ ]:
# Execute remove error rows in data files
#directory = "datos_con_error/"
#list_of_csv = list_reports(directory,"csv")

#for file_csv in list_of_csv:
#    year = file_csv[len(file_csv)-6:len(file_csv)-4]
#    data_file = f"../DATA/Bronce/COVID19MEXICO20{year}/COVID19MEXICO20{year}.csv"
#    new_data_file = f"../DATA/Silver/COVID19MEXICO20{year}/COVID19MEXICO20{year}.csv"
#    remove_list_file = f"datos_con_error/errores_COVID19MEXICO20{year}.csv"
#    remove_list = pd.read_csv(remove_list_file)['ID_REGISTRO']
#    clean_csv_for_batch(data_file,new_data_file,remove_list)
#    file_csv.rename(file_csv.replace("datos_con_error","datos_limpiados"))

Clean file: COVID19MEXICO2022/COVID19MEXICO2022.csv
Clean file: COVID19MEXICO2020/COVID19MEXICO2020.csv
Clean file: COVID19MEXICO2024/COVID19MEXICO2024.csv
